# E-HEDO + X-HVSC on native Mamba-2: reproducible Flickr8k pipeline

Implements the proposal *Hamiltonian-Inspired Energy Dissipation and Chunk-wise Variational State Coupling for
Efficient Multimodal State Space Models*. `hedo_native/METHOD.md` has the formulation, **Theorem 1 (energy
non-increase)** with proof, the complexity analysis, and the **H1-H4 criteria fixed before any run**.

This notebook only orchestrates. All methodology is in version-controlled files under `Native mamba notebook/hedo_native/`.
Every artifact records the code hash, git commit, config hash, and dataset/feature checksums. A run counts only if
`run_summary.json` says `COMPLETED`: finite training, checkpoint reload reproduces validation, deterministic inference
(including exchange scores), and an independent evaluator reproduces the test metrics (including re-ranking).

| Step | Cell | Pass condition |
|---|---|---|
| Environment, code checkout, unit tests | 0-2 | `NATIVE_MAMBA2_CHECK: PASS`, tests ok |
| Data + frozen features (+ ViT saliency) | 3-4 | `DATASET_READY`, `FEATURE_CACHE` |
| Gate: Theorem 1 numerics, exchange no-op at init, all variants | 5 | `GATE: PASS` |
| Sanity: the proposed model learns (2 epochs) | 6 | val MR well above chance (~0.53) |
| Proposed 2x2 factorial x seeds | 7 | all `COMPLETED` |
| Ablations and controls | 8 | all `COMPLETED` |
| H1/H2 probes | 9 | `PROBES: PASS` |
| H3 corruption benchmark | 10 | `ROBUSTNESS_EVAL: PASS` |
| H4 scaling + end-to-end efficiency | 11 | `SCALING`, `EFFICIENCY_PROFILE` |
| Tables, statistics, hypothesis verdicts | 12 | `ANALYSIS`, `HYPOTHESES` |
| Bundle | 13 | |

Verdicts come from the fixed criteria. "Not supported" is a valid, reportable outcome.

In [ ]:
# CELL 0: CONFIGURATION (the only cell you should edit)
import os, subprocess, sys, time, json
import pandas as pd
from pathlib import Path

REPO_URL = "https://github.com/abhinavsai2006/PH_SSD.git"   # private repo: use a token URL or upload the folder
REPO_REF = "main"                    # pin to a commit hash for the final paper runs
CODE_ROOT = Path("/content/PH_SSD")
PKG = CODE_ROOT / "Native mamba notebook" / "hedo_native"
NATIVE_PYTHON = "/content/mamba312/bin/python"
WORK = Path("/content/hedo_work")    # data, features, runs, report (mount Drive here to survive disconnects)

CORE_SEEDS = [42, 43, 44, 45, 46]    # 5 seeds: n=3 gives df=2 and no usable Wilcoxon test
ABLATION_SEEDS = [42, 43, 44, 45, 46]
EPOCHS = 10

os.environ["HEDO_WORK"] = str(WORK)
os.environ["PYTHONUNBUFFERED"] = "1"
WORK.mkdir(parents=True, exist_ok=True)
(WORK / "logs").mkdir(exist_ok=True)


def run(cmd, log_name, cwd=None, check=True):
    # Stream a native subprocess live and tee it to WORK/logs/<log_name>.log
    log_path = WORK / "logs" / f"{log_name}.log"
    print("$", " ".join(map(str, cmd)), flush=True)
    with open(log_path, "w", encoding="utf-8") as log:
        p = subprocess.Popen(list(map(str, cmd)), cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1, env=os.environ.copy())
        for line in p.stdout:
            print(line, end="", flush=True)
            log.write(line)
        rc = p.wait()
    print(f"[exit {rc}] log: {log_path}", flush=True)
    if check and rc != 0:
        raise RuntimeError(f"{log_name} failed with exit code {rc}; see {log_path}")
    return rc


print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

In [ ]:
# CELL 1: NATIVE PYTHON 3.12 ENVIRONMENT (pinned)
import shutil

TORCH = ["torch==2.9.0", "torchvision==0.24.0"]
PINNED = ["numpy==2.1.3", "pandas==2.2.3", "scipy==1.14.1", "matplotlib==3.9.2", "Pillow==11.0.0",
          "transformers==4.56.2", "einops==0.8.1", "packaging", "ninja", "pytest"]
WHEELS = [
    "https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.2.post1/"
    "causal_conv1d-1.6.2.post1+cu12torch2.9cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
    "https://github.com/state-spaces/mamba/releases/download/v2.3.2.post1/"
    "mamba_ssm-2.3.2.post1+cu12torch2.9cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
]

if not Path(NATIVE_PYTHON).is_file():
    run([sys.executable, "-m", "pip", "install", "-q", "uv"], "install_uv")
    run(["uv", "venv", "--python", "3.12", "/content/mamba312"], "venv")
    run(["uv", "pip", "install", "--python", NATIVE_PYTHON, "pip"], "install_pip")
    run(["uv", "pip", "install", "--python", NATIVE_PYTHON, *TORCH,
         "--index-url", "https://download.pytorch.org/whl/cu128"], "install_torch")
    run(["uv", "pip", "install", "--python", NATIVE_PYTHON, *PINNED], "install_pinned")
    run(["uv", "pip", "install", "--python", NATIVE_PYTHON, "--no-deps", *WHEELS], "install_mamba")

check = r'''
import json, sys, torch
from mamba_ssm import Mamba2
assert torch.cuda.is_available()
assert Mamba2.__module__ == "mamba_ssm.modules.mamba2", Mamba2.__module__
m = Mamba2(d_model=128, d_state=64, d_conv=4, expand=2, headdim=64).cuda()
x = torch.randn(2, 64, 128, device="cuda", requires_grad=True)
y = m(x); y.sum().backward()
assert y.shape == x.shape and torch.isfinite(y).all() and torch.isfinite(x.grad).all()
print("torch", torch.__version__, "cuda", torch.version.cuda, torch.cuda.get_device_name(0),
      torch.cuda.get_device_capability(0))
print("NATIVE_MAMBA2_CHECK: PASS")
'''
run([NATIVE_PYTHON, "-c", check], "native_check")
freeze = subprocess.run([NATIVE_PYTHON, "-m", "pip", "freeze"], capture_output=True, text=True).stdout.splitlines()
json.dump({"pip_freeze": freeze, "pinned": TORCH + PINNED, "wheels": WHEELS},
          open(WORK / "environment_install.json", "w"), indent=2)

In [ ]:
# CELL 2: CODE CHECKOUT + CPU UNIT TESTS
if not CODE_ROOT.exists():
    run(["git", "clone", REPO_URL, CODE_ROOT], "git_clone")
run(["git", "fetch", "--all"], "git_fetch", cwd=CODE_ROOT)
is_branch = subprocess.run(["git", "rev-parse", "--verify", "-q", f"origin/{REPO_REF}"], cwd=CODE_ROOT,
                           capture_output=True).returncode == 0
run(["git", "checkout", "--detach", f"origin/{REPO_REF}" if is_branch else REPO_REF], "git_checkout", cwd=CODE_ROOT)
run(["git", "log", "-1", "--format=%H %s"], "git_head", cwd=CODE_ROOT)
assert PKG.is_dir(), PKG

run([NATIVE_PYTHON, "-m", "pytest", "-q", "tests/test_cpu.py"], "unit_tests", cwd=PKG)
run([NATIVE_PYTHON, "-c", "from common import code_sha256, git_commit; print('code_sha256', code_sha256()); "
     "print('git', git_commit())"], "code_hash", cwd=PKG)

In [ ]:
# CELL 3: FLICKR8K (download, SHA256, official splits, leakage report)
# First run records archive checksums in manifest.json. Save {"Flickr8k_Dataset.zip": sha, "Flickr8k_text.zip": sha}
# as hedo_native/flickr8k_checksums.json and commit it: later runs then enforce them.
CHECKSUMS = PKG / "flickr8k_checksums.json"
args = ["--expected-checksums", CHECKSUMS] if CHECKSUMS.is_file() else []
run([NATIVE_PYTHON, "data.py", *args], "data", cwd=PKG)
m = json.load(open(WORK / "data/flickr8k/manifest.json"))
print(json.dumps({k: m[k] for k in ("archive_sha256", "counts", "leakage")}, indent=2))

In [ ]:
# CELL 4: FROZEN BACKBONE FEATURE CACHE (about 6 GB; rebuilt automatically if feature code changed)
import shutil
# Official ViT-B/16 IMAGENET1K_V1 transforms, FP32 compute, FP16 storage with overflow check.
# RoBERTa commit hash and loading report are saved. Set HEDO_ROBERTA_REVISION to that hash for the paper runs.
code_now = subprocess.run([NATIVE_PYTHON, "-c", "from common import feature_code_sha256; print(feature_code_sha256())"],
                          cwd=PKG, capture_output=True, text=True, check=True).stdout.strip()
fm_path = WORK / "features/feature_manifest.json"
cached = json.load(open(fm_path)).get("feature_code_sha256") if fm_path.is_file() else None
if cached != code_now or not (WORK / "features/saliency_test.npy").is_file():
    print(f"feature cache {'missing' if cached is None else 'stale'} -> rebuilding (feature code {code_now[:12]})")
    shutil.rmtree(WORK / "features", ignore_errors=True)
    run([NATIVE_PYTHON, "features.py", "--batch-size", "64"], "features", cwd=PKG)
else:
    print(f"feature cache current (feature code {code_now[:12]})")
fm = json.load(open(WORK / "features/feature_manifest.json"))
print("RoBERTa:", fm["roberta"], "| absmax:", fm["absmax"])
print(open(WORK / "features/roberta_loading_info.json").read()[:800])

In [ ]:
# CELL 5: PRE-TRAINING GATE ON REAL DATA
run([NATIVE_PYTHON, "gate.py"], "gate", cwd=PKG)
print(open(WORK / "report/param_counts.json").read())

In [ ]:
# CELL 6: LEARNING SANITY (decision point before any matrix run)
# 2 epochs, per-batch JSONL, anomaly detection in epoch 1, exit 3 if epoch-1 val MR < 2.0 (chance ~0.53).
# Compare the proposed model with the no-mixer control. If ours is not clearly above chance, stop and debug.
SANITY = WORK / "sanity"
for variant in ("no_mixer_meanpool", "full_x"):
    out = SANITY / f"{variant}_seed42"
    run([NATIVE_PYTHON, "-u", "train.py", "--variant", variant, "--seed", 42, "--epochs", 2, "--detect-anomaly",
         "--out", out, "--overwrite"], f"sanity_{variant}", cwd=PKG, check=False)
    print(variant, json.load(open(out / "run_summary.json"))["status"])
    if (out / "training_history.csv").is_file():
        h = pd.read_csv(out / "training_history.csv")
        keep = ("epoch", "train_loss", "train_infonce", "train_infonce_exchange", "train_prior_kl",
                "val_mean_recall", "val_dual_mean_recall")
        print(h[[c for c in h.columns if c in keep]].to_string(index=False))
log = pd.read_json(SANITY / "full_x_seed42" / "batch_log.jsonl", lines=True)
print(log[[c for c in log.columns if "energy" in c or "gate" in c or c.startswith("grad_")]].describe().T.to_string())

# HARD STOP: the matrix costs many GPU hours; never start it on a model that does not train.
ALLOW_MATRIX_WITHOUT_SANITY = False
summary = json.load(open(SANITY / "full_x_seed42" / "run_summary.json"))
hist_path = SANITY / "full_x_seed42" / "training_history.csv"
val_mr = float(pd.read_csv(hist_path)["val_mean_recall"].max()) if hist_path.is_file() else float("nan")
sanity_ok = summary["status"] == "COMPLETED" and val_mr >= 5.0
print(f"SANITY: status={summary['status']} best val MR={val_mr:.3f} (chance ~0.53, required >= 5.0) ->",
      "PASS" if sanity_ok else "FAIL")
if not sanity_ok and not ALLOW_MATRIX_WITHOUT_SANITY:
    raise RuntimeError("Sanity run failed. Inspect WORK/logs/sanity_full_x.log and "
                       "WORK/sanity/full_x_seed42/failure_state.pt before running cells 7+.")

In [ ]:
# CELL 7: PROPOSED 2x2 FACTORIAL (baseline, +E-HEDO, +X-HVSC, +both) x CORE_SEEDS
# Safe to re-run after a disconnect: verified current runs are skipped, anything else restarts.
run([NATIVE_PYTHON, "-u", "run_matrix.py", "--suite", "proposal", "--epochs", EPOCHS, "--seeds", *CORE_SEEDS],
    "matrix_proposal", cwd=PKG, check=False)
print(pd.read_csv(WORK / "results/all_runs.csv")[["run", "status", "mean_recall", "dual_mean_recall"]].to_string(index=False))

In [ ]:
# CELL 8: ABLATIONS AND CONTROLS (identical protocol)
#   full_x_affine             E-HEDO replaced by the earlier affine HEDO
#   full_x_constdamp          damping not input-dependent (tests the background-selective mechanism)
#   full_x_noexchange         bottleneck only, no cross-modal messages (H2 reference)
#   full_x_noprior            beta = 0 (no information bottleneck)
#   baseline_param_matched_x  Mamba-2 with a wider head, #params matched to ours
#   transformer_param_matched Transformer mixer, #params matched to baseline
#   transformer_x             our modules on a Transformer mixer
#   no_mixer_meanpool         frozen features + mean pool
#   proposal_chunks           chunk size 8 and 32
RUN_LEGACY = False   # True also reruns the earlier affine-HEDO / index-KL study
suites = ["proposal_ablations", "proposal_chunks"] + (["core", "ablations", "klsweep", "chunks"] if RUN_LEGACY else [])
run([NATIVE_PYTHON, "-u", "run_matrix.py", "--suite", *suites, "--epochs", EPOCHS, "--seeds", *ABLATION_SEEDS],
    "matrix_ablations", cwd=PKG, check=False)
df = pd.read_csv(WORK / "results/all_runs.csv")
print(df.groupby(["variant", "hvsc_chunk_size", "status"]).size().to_string())

In [ ]:
# CELL 9: H1 / H2 MECHANISM PROBES (energy vs saliency, occlusion, gradient balance, message reliance)
run([NATIVE_PYTHON, "-u", "probe.py"], "probes", cwd=PKG)
pr = pd.read_csv(WORK / "results/probes.csv")
cols = ["attenuation_bg", "attenuation_fg", "attenuation_spearman_saliency", "energy_max_increase_float64",
        "occlude_bg_dual_mean_recall", "occlude_fg_dual_mean_recall", "grad_abs_log10_ratio", "dominance_index"]
print(pr.groupby("variant")[[c for c in cols if c in pr]].mean().round(4).to_string())

In [ ]:
# CELL 10: H3 CORRUPTION BENCHMARK (image noise/blur/JPEG/occlusion; caption dropout/typos/shuffle)
if not (WORK / "features/robust/robust_manifest.json").is_file():
    run([NATIVE_PYTHON, "-u", "robustness.py", "build"], "robust_build", cwd=PKG)
run([NATIVE_PYTHON, "-u", "robustness.py", "evaluate"], "robust_eval", cwd=PKG)
rb = pd.read_csv(WORK / "results/robustness.csv")
print(rb.groupby(["variant", "corruption"])["relative_mr"].mean().unstack().round(3).to_string())

In [ ]:
# CELL 11: H4 SCALING WITH SEQUENCE LENGTH + END-TO-END EFFICIENCY (real inputs, backbones included)
run([NATIVE_PYTHON, "-u", "scaling.py"], "scaling", cwd=PKG)
print(pd.read_csv(WORK / "report/scaling/scaling.csv").to_string(index=False))
run([NATIVE_PYTHON, "-u", "profile_efficiency.py", "--batch-sizes", 1, 32], "efficiency", cwd=PKG)
print(pd.read_csv(WORK / "report/efficiency/efficiency_profile.csv").T.to_string())

In [ ]:
# CELL 12: TABLES, STATISTICS, FIGURES, HYPOTHESIS VERDICTS (only verified, current-code runs)
from IPython.display import Image, display
run([NATIVE_PYTHON, "-u", "analyze.py", "--bootstrap", 2000], "analyze", cwd=PKG)
run([NATIVE_PYTHON, "-u", "hypotheses.py"], "hypotheses", cwd=PKG)
for name in ("table_main.tex", "table_ablations.tex", "table_stats.tex", "table_hypotheses.tex"):
    print(f"--- {name}\n" + (WORK / "report" / name).read_text())
for fig in ("fig_mean_recall.png", "fig_training_curves.png"):
    display(Image(str(WORK / "report" / fig)))

In [ ]:
# CELL 13: BUNDLE (small artifacts + SHA256SUMS; checkpoints and features excluded by default)
import hashlib, zipfile
INCLUDE_CHECKPOINTS = False
bundle = WORK / "hedo_hvsc_bundle.zip"
files = [p for p in WORK.rglob("*") if p.is_file() and p != bundle and "features" not in p.parts
         and "extracted" not in p.parts and p.suffix not in {".zip", ".part"}
         and (INCLUDE_CHECKPOINTS or p.suffix != ".pt")]
sums = "".join(f"{hashlib.sha256(p.read_bytes()).hexdigest()}  {p.relative_to(WORK)}\n" for p in sorted(files))
(WORK / "SHA256SUMS").write_text(sums)
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(files) + [WORK / "SHA256SUMS"]:
        z.write(p, p.relative_to(WORK))
print(bundle, f"{bundle.stat().st_size / 2**20:.1f} MB", len(files), "files")